In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="quora_cross_encoder_softmax_threshold_065_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [1]:
import torch
import numpy as np
from datasets import Dataset
from sentence_transformers import CrossEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report


def softmax_np(x, axis=-1):
    x = np.asarray(x)
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [2]:
model_name = "cross-encoder/quora-distilroberta-base"
model = CrossEncoder(model_name, device=str(device))
print(model_name)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/quora-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


cross-encoder/quora-distilroberta-base


In [3]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])
pairs = list(zip(sent1, sent2))

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [4]:
batch_size = 64
threshold = 0.65

raw_outputs = model.predict(
    pairs,
    batch_size=batch_size,
    show_progress_bar=True,
    convert_to_numpy=True,
)

raw_outputs = np.asarray(raw_outputs)
print("raw_outputs.shape:", raw_outputs.shape)

if raw_outputs.ndim == 2 and raw_outputs.shape[1] == 2:
    pos_scores = softmax_np(raw_outputs, axis=1)[:, 1]
elif raw_outputs.ndim == 2 and raw_outputs.shape[1] == 1:
    pos_scores = raw_outputs[:, 0]
else:
    pos_scores = raw_outputs.reshape(-1)

y_pred = (pos_scores >= threshold).astype(int)
print("done")


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

raw_outputs.shape: (408,)
done


In [ ]:

vault.create_record_list("distilbert_quora_softmax_prediction", column_names=["y_pred", "pos_scores"])

for i in range(len(y_pred)):
    vault.append_record("distilbert_quora_softmax_prediction", 
                        {
                            "y_pred": y_pred[i],
                            "pos_scores": float(pos_scores[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "Per-example prediction dataset produced by running the cross-encoder model cross-encoder/quora-distilroberta-base on the glue_mrpc_validation sentence-pair dataset. Each record corresponds to one MRPC validation example and stores two fields: y_pred, the final binary paraphrase prediction after applying a 0.65 threshold, and pos_scores, the model\u2019s positive-class score/probability for the paraphrase label derived from the model output. The records are linked back to the matching source example in glue_mrpc_validation, enabling row-level inspection of predictions, error analysis, and downstream aggregation. In this workflow, this dataset is the main prediction output used to compare model decisions against ground-truth labels and to build the evaluation summary dataset."
embedding = get_embeddings(description)
vault.create_description("distilbert_quora_softmax_prediction", description, embedding)

properties = {"task": "paraphrase detection", "dataset_role": "model predictions", "source": "glue/mrpc", "split": "validation", "model": "cross-encoder/quora-distilroberta-base", "input_type": "sentence pair", "score_type": "softmax positive-class score", "threshold": "0.65", "output_columns": "y_pred,pos_scores", "labels": "not_paraphrase,paraphrase", "domain": "news", "size": "408"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_quora_softmax_prediction", cat, embedding, prop)

In [5]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=['not_paraphrase', 'paraphrase'])

print({"accuracy": acc, "f1": f1, "threshold": threshold})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.678921568627451, 'f1': 0.7578558225508318, 'threshold': 0.65}
                precision    recall  f1-score   support

not_paraphrase       0.49      0.56      0.52       129
    paraphrase       0.78      0.73      0.76       279

      accuracy                           0.68       408
     macro avg       0.64      0.65      0.64       408
  weighted avg       0.69      0.68      0.68       408



In [6]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]))
    print("positive_score:", float(pos_scores[i]))
    print("pred:", int(y_pred[i]), "label:", "paraphrase" if int(y_pred[i]) == 1 else "not_paraphrase")


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1
positive_score: 0.9779019355773926
pred: 1 label: paraphrase
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0
positive_score: 0.006291278637945652
pred: 0 label: not_paraphrase
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0
positive_score: 0.04939701780676842
pred: 0 label: not_paraphrase
sentence1: The AFL-CIO is waiting until October to decide 

In [7]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "positive_score:", float(pos_scores[i]))


num_errors: 131
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
true: 1 pred: 0 positive_score: 0.015424183569848537
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
sentence2: The Institute said dioxin levels in the environment have fallen by as much as 76 percent since the 1970s .
true: 0 pred: 1 positive_score: 0.8578875660896301
idx: 7
sentence1: This integrates with Rational PurifyPlus and allows developers to work in supported versions of Java , Visual C # and Visual Basic .NET.
sentence2: IBM said the Rational products were also integrated with Rational PurifyPlus , which allows developers to work in Java , Visual C # and VisualBasic .Net.
true: 1 pred: 0 positive_score: 0.2543953359

In [8]:
vault.create_record_list("quora_cross_encoder_softmax_threshold_065_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])

summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("quora_cross_encoder_softmax_threshold_065_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "distilbert_quora_softmax_prediction": [0, len(ds)]
                    })

summary

description = "Experiment-level evaluation summary for the notebook run using the cross-encoder/quora-distilroberta-base model on the GLUE MRPC validation split with a paraphrase decision threshold of 0.65 applied to the positive softmax score. This dataset contains a single summary record with the fields: accuracy (float), f1 (float), and classification_report (string produced by sklearn, summarizing precision, recall, f1-score, and support for the not_paraphrase and paraphrase classes). It is derived from the full glue_mrpc_validation dataset and the corresponding per-example predictions stored in distilbert_quora_softmax_prediction, and its role is to capture the overall performance of this experiment in a compact, queryable form for comparison, tracking, and documentation."
embedding = get_embeddings(description)
vault.create_description("quora_cross_encoder_softmax_threshold_065_mrpc_summary", description, embedding)

properties = {"artifact_type": "evaluation summary", "task": "paraphrase detection", "model": "cross-encoder/quora-distilroberta-base", "dataset": "GLUE MRPC", "source": "glue_mrpc_validation", "split": "validation", "size": "408", "input_type": "sentence pair", "scoring": "softmax positive class score", "threshold": "0.65", "metrics": "accuracy,f1,classification_report", "label_space": "not_paraphrase,paraphrase"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("quora_cross_encoder_softmax_threshold_065_mrpc_summary", cat, embedding, prop)

{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'cross-encoder/quora-distilroberta-base',
 'device': 'mps',
 'num_examples': 408,
 'threshold': 0.65,
 'accuracy': 0.678921568627451,
 'f1': 0.7578558225508318}

In [ ]:
description = "quora_cross_encoder_softmax_threshold_065_mrpc is the experiment-level dataset for evaluating the cross-encoder/quora-distilroberta-base model on the GLUE MRPC validation split using a paraphrase decision threshold of 0.65. It links three parts of the workflow: the input dataset glue_mrpc_validation, which provides sentence pairs and gold labels (sentence1, sentence2, label); the per-example prediction records in distilbert_quora_softmax_prediction, which store y_pred and pos_scores for each MRPC example; and the aggregate evaluation record in quora_cross_encoder_softmax_threshold_065_mrpc_summary, which stores accuracy, f1, and classification_report over the full validation set. Its role is to document the full prediction-and-evaluation run, including model outputs, thresholded binary decisions, and final performance metrics for paraphrase detection." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("quora_cross_encoder_softmax_threshold_065_mrpc", description, embedding)

properties = {"artifact_type": "model evaluation run", "task": "paraphrase detection", "model": "cross-encoder/quora-distilroberta-base", "model_family": "cross-encoder", "scoring": "softmax positive-class score", "threshold": "0.65", "source": "glue/mrpc", "benchmark": "GLUE MRPC", "split": "validation", "size": "408", "input_type": "sentence pair", "labels": "not_paraphrase|paraphrase", "domain": "news", "outputs": "predicted labels, positive scores, accuracy, f1, classification report"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("quora_cross_encoder_softmax_threshold_065_mrpc", cat, embedding, prop)